# Model Training

## Load Feature Matrix

In [9]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

features_df = pd.read_csv(os.path.join('..', 'Feature_Engineering', 'features.csv'))

print("Shape:", features_df.shape)
features_df.head()

Shape: (8068, 14)


,date,home_team,away_team,home_form,away_form,form_diff,home_avg_scored,home_avg_conceded,away_avg_scored,away_avg_conceded,goal_diff,is_neutral,weight,result
0,2018-01-02,Iraq,United Arab Emirates,0.5,0.5,0.0,1.0,1.0,1.0,1.0,0.0,1,1.0,0
1,2018-01-02,Oman,Bahrain,0.5,0.5,0.0,1.0,1.0,1.0,1.0,0.0,1,1.0,1
2,2018-01-05,Oman,United Arab Emirates,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1,1.0,0
3,2018-01-07,Estonia,Sweden,0.5,0.5,0.0,1.0,1.0,1.0,1.0,0.0,1,0.5,0
4,2018-01-11,Denmark,Sweden,0.5,0.0,0.5,1.0,1.0,1.0,1.0,0.0,1,0.5,-1


## Prepare Training Data

In [10]:
# Define features (X) and target (y)
# These are the columns the model learns from
feature_cols = [
    'home_form', 'away_form', 'form_diff',
    'home_avg_scored', 'home_avg_conceded',
    'away_avg_scored', 'away_avg_conceded',
    'goal_diff', 'is_neutral'
]

X = features_df[feature_cols]
y = features_df['result']
weights = features_df['weight']

# Split into training and testing sets (80/20)
# 80% to train the model, 20% to test how good it is
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

Training set size: (6454, 9)
Testing set size: (1614, 9)


## Train Logistic Regression (Baseline)

In [11]:
# Logistic Regression — our simple baseline model
# We scale the data first so all features are on the same scale

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train, sample_weight=w_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_preds)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

Logistic Regression Accuracy: 0.5211


## Train XGBoost (Main Model)

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train_enc, sample_weight=w_train)

xgb_preds = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test_enc, xgb_preds)

print(f"XGBoost Accuracy: {xgb_accuracy:.4f}")
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

XGBoost Accuracy: 0.5093
Label mapping: {np.int64(-1): np.int64(0), np.int64(0): np.int64(1), np.int64(1): np.int64(2)}


## Compare Models

In [15]:
print("=" * 35)
print("        MODEL COMPARISON")
print("=" * 35)
print(f"Logistic Regression : {lr_accuracy:.4f}")
print(f"XGBoost             : {xgb_accuracy:.4f}")
print("=" * 35)

        MODEL COMPARISON
Logistic Regression : 0.5211
XGBoost             : 0.5093


## Save Models

In [16]:
import pickle

with open(os.path.join('..', 'Model_Training', 'xgb_model.pkl'), 'wb') as f:
    pickle.dump(xgb_model, f)

with open(os.path.join('..', 'Model_Training', 'lr_model.pkl'), 'wb') as f:
    pickle.dump(lr_model, f)

with open(os.path.join('..', 'Model_Training', 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

with open(os.path.join('..', 'Model_Training', 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

print("All models saved!")

All models saved!
